In [ ]:
# -------------------------------------------------------------
# Common pre‑amble – shared across all notebooks
# -------------------------------------------------------------
from config.notebook_setup import *


# Reuters News Topic Classification - Model Training

This notebook implements and trains different models for the Reuters news topic classification task. We'll compare several approaches:

1. Multinomial Naive Bayes with TF-IDF
2. Linear SVM with TF-IDF (unigrams)
3. Linear SVM with TF-IDF (bigrams)
4. MiniLM embeddings with Logistic Regression
5. RAG-kMajority
6. RAG-CentroidNN
7. RAG-LLM

## Setup and Data Loading

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import logging
import warnings
import random
import os
import numpy as np
from src.datasets.dataset import load_data
from src.algorithms.naive_bayes import NaiveBayesClassifier
from src.algorithms.linear_svm import LinearSVMClassifier, LinearSVMBigrams
from src.algorithms.transformer_logreg import TransformerLogReg
from src.rag import load_kmajority, load_centroid, load_llm
from src.rag.adapter_sklearn import RagSklearnAdapter
from src.evaluation import run_evaluations
from src.embeddings.openai_embedder import OpenAIEmbedder
from src.rag.vector_store import VectorStore

# Disable HuggingFace tokenizers parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load dataset
N_CLASSES = 10
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
print(f'Train docs: {len(X_train):,},  Test docs: {len(X_test):,}')
print('Labels:', label_names)

/home/marcmaceira/venv/reuters-rag-classifier/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-04 01:35:23,812 - faiss.loader - INFO - Loading faiss with AVX2 support.
2025-05-04 01:35:23,841 - faiss.loader - INFO - Successfully loaded faiss with AVX2 support.
2025-05-04 01:35:23,849 - faiss - INFO - Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


Train docs: 6,337,  Test docs: 2,477
Labels: ['earn', 'acq', 'crude', 'interest', 'money-fx', 'trade', 'grain', 'corn', 'dlr', 'money-supply']


## Initialize and Train All Models

In [3]:
# ensure we’re using OpenAI for embeddings
os.environ['USE_OPENAI_EMBEDDINGS'] = '1'
# OR explicitly pass use_openai=True below

# instantiate the OpenAI embedder (batch size adjustable)
openai_embedder = OpenAIEmbedder(model="text-embedding-3-small", batch_size=50)


In [4]:
# Initialize all models
# Initialize all models
models = {
    'Naive Bayes':       NaiveBayesClassifier(),
    'Linear SVM':        LinearSVMClassifier(),
    'TF-IDF bigrams + SVM': LinearSVMBigrams(),
    'MiniLM + LogReg':   TransformerLogReg(),
    # RAG variants all take an embedder under the hood:
    'RAG-kMajority':     RagSklearnAdapter(load_kmajority(top_k=5)),
    'RAG-CentroidNN':    RagSklearnAdapter(load_centroid()),
    # For the LLM‐backed RAG we also pass the same embedder plus your LLM choice:
    'RAG-LLM (OpenAI-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            embedder=openai_embedder,
            use_openai=True  # Add this parameter to use the OpenAI index
        )
    ),
    # Add the local embeddings variant:
    'RAG-LLM (local-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            use_openai=False,  # Explicitly specify to use local index
            embedder=lambda texts: VectorStore.embed("sentence-transformers/all-MiniLM-L6-v2", texts)
        )
    ),
}

2025-05-04 01:35:26,041 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2025-05-04 01:35:26,042 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2025-05-04 01:35:28,201 - src.rag.retrieval - INFO - Loading SentenceTransformer retriever
2025-05-04 01:35:28,202 - src.rag.retrieval - INFO - Using index: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 01:35:28,203 - src.rag.vector_store - INFO - Loading FAISS index from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss
2025-05-04 01:35:28,219 - src.rag.vector_store - INFO - Loading metadata from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 01:35:29,84

OPENAI_API_KEY loaded successfully.


2025-05-04 01:35:32,721 - src.rag.vector_store - INFO - VectorStore initialized with 6337 documents in 1.30 seconds
2025-05-04 01:35:32,723 - src.rag.retrieval - INFO - Default retriever loaded in 1.31 seconds
2025-05-04 01:35:32,724 - src.rag.retrieval - INFO - Loading SentenceTransformer retriever
2025-05-04 01:35:32,727 - src.rag.retrieval - INFO - Using index: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 01:35:32,728 - src.rag.vector_store - INFO - Loading FAISS index from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/index.faiss
2025-05-04 01:35:32,738 - src.rag.vector_store - INFO - Loading metadata from /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/artifacts/sbert/meta.jsonl
2025-05-04 01:35:34,125 - src.rag.vector_store - 

In [5]:
from src.training import run_trainings
from src.evaluation import run_evaluations
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC


# ▸  train + persist
trained = run_trainings(models, X_train, y_train,
                        output_dir="artifacts/models")




Training:   0%|          | 0/8 [00:00<?, ?it/s]

Training:  12%|█▎        | 1/8 [00:01<00:11,  1.65s/it]

Saved model to artifacts/models/Naive Bayes.joblib


Training:  25%|██▌       | 2/8 [00:03<00:11,  1.86s/it]

Saved model to artifacts/models/Linear SVM.joblib


Training:  38%|███▊      | 3/8 [00:09<00:17,  3.56s/it]2025-05-04 01:35:43,389 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2025-05-04 01:35:43,390 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Saved model to artifacts/models/TF-IDF bigrams + SVM.joblib


2025-05-04 01:35:44,808 - src.algorithms.transformer_logreg - INFO - Encoding 6337 documents for training
Training: 100%|██████████| 8/8 [03:30<00:00, 26.27s/it]

Saved model to artifacts/models/MiniLM + LogReg.joblib
Saved model to artifacts/models/RAG-kMajority.joblib
Saved model to artifacts/models/RAG-CentroidNN.joblib
Saved model to artifacts/models/RAG-LLM (OpenAI-embeddings).joblib
Saved model to artifacts/models/RAG-LLM (local-embeddings).joblib


In [7]:
# Dictionary to collect all evaluation results
all_results = {}

# Evaluate each model and collect results with enhanced logging
for name, model in trained.items():
    result = run_evaluations(
        model,                 # or f"artifacts/models/{name}.joblib"
        X_test, y_test,
        X_train=X_train, y_train=y_train,
        output_dir=f"artifacts/results/{name}",
        model_name=name  # Pass the model name for better logging
    )
    all_results[name] = result

2025-05-04 01:39:04,347 - root - INFO - 🔍 Evaluating model: Naive Bayes
2025-05-04 01:39:04,349 - root - INFO -   • Evaluating on test set (2477 samples)...
2025-05-04 01:39:04,711 - root - INFO -     ✓ Test accuracy: 0.9281
2025-05-04 01:39:04,757 - root - INFO -     ✓ Test macro F1: 0.8216, weighted F1: 0.9277
2025-05-04 01:39:05,142 - root - INFO -     ✓ Saved test reports to artifacts/results/Naive Bayes
2025-05-04 01:39:05,143 - root - INFO -   • Evaluating on train set (6337 samples)...
2025-05-04 01:39:05,813 - root - INFO -     ✓ Train accuracy: 0.9604
2025-05-04 01:39:05,880 - root - INFO -     ✓ Train macro F1: 0.9350, weighted F1: 0.9601
2025-05-04 01:39:06,241 - root - INFO -     ✓ Saved train reports to artifacts/results/Naive Bayes
2025-05-04 01:39:06,242 - root - INFO -   ✓ Evaluation of Naive Bayes completed
2025-05-04 01:39:06,244 - root - INFO - 🔍 Evaluating model: Linear SVM
2025-05-04 01:39:06,245 - root - INFO -   • Evaluating on test set (2477 samples)...
2025-05-

In [ ]:

# After all evaluations, print a comparison summary
logging.info("📊 Model Performance Comparison:")
for name, result in all_results.items():
    test_acc = result['test']['accuracy']
    test_f1 = result['test']['macro_f1']
    
    if 'train' in result:
        train_acc = result['train']['accuracy']
        train_f1 = result['train']['macro_f1']
        logging.info(f"{name:20} | Test acc: {test_acc:.4f}, f1: {test_f1:.4f} | Train acc: {train_acc:.4f}, f1: {train_f1:.4f}")
    else:
        logging.info(f"{name:20} | Test acc: {test_acc:.4f}, f1: {test_f1:.4f} | Train: N/A")

## Model Comparison

In [11]:

# Organize results using our new function
from src.evaluation import collect_evaluation_results, compare_models, print_evaluation_results
overall_results, class_results = collect_evaluation_results(all_results)

# Create a DataFrame for comparison
results_df = compare_models(overall_results)
print(results_df)

# Use our utility functions to organize and print results
print_evaluation_results(overall_results, class_results)

             Accuracy  Macro F1  Weighted F1
Naive Bayes     0.928     0.822        0.928
Linear SVM      0.947     0.867        0.947
Overall Model Comparison:
             accuracy  macro_f1  weighted_f1  macro_precision  macro_recall  \
Naive Bayes     0.928     0.822        0.928            0.854         0.808   
Linear SVM      0.947     0.867        0.947            0.874         0.862   

             weighted_precision  weighted_recall  
Naive Bayes               0.931            0.928  
Linear SVM                0.947            0.947  

Per-Class Model Comparison:

Class: acq
             precision  recall  f1-score  support
Naive Bayes      0.958   0.975     0.966    719.0
Linear SVM       0.974   0.976     0.975    719.0

Class: corn
             precision  recall  f1-score  support
Naive Bayes      0.868   0.688     0.767     48.0
Linear SVM       0.949   0.771     0.851     48.0

Class: crude
             precision  recall  f1-score  support
Naive Bayes      0.916   0.962

## Save model and metrics

In [9]:
# Save all models
import joblib
import os
from src.utils.model_storage import save_model

# Save all models
for name, model in models.items():
    save_model(
        model=model,
        model_name=name,
        metrics=results[name],
        n_classes=N_CLASSES
    )

NameError: name 'results' is not defined

## Next Steps

In the next notebook, we'll:
1. Perform detailed error analysis
2. Create a confusion matrix to understand model mistakes
3. Implement a semantic search demo using the transformer embeddings